# Kaggriculture C95 — Current-Meta High-Score Agent with ADL Telemetry

This notebook packages the exact frozen **C95 top-20 wheat counter** selected by
the supplied C92→C94→C95 tournament lineage. It does not approximate or rewrite
the policy. The submitted `main.py` must retain SHA-256
`489f5d197527f107027626cce79d850fd2ca90edd43d94384b849b6511e27bdb`.

The notebook is self-contained: it embeds C95 and the C94 veto control, requires
no model or replay input, builds a deterministic `submission.tar.gz`, verifies
the exact file-path entrypoint, runs Kaggriculture smoke/paired validation when
the competition engine is available, and emits real ADL transition rows from
observed games. Instrumentation stays outside `main.py`, so it cannot alter the
submitted policy.

Evidence boundary: C95 led C94 112-8 versus 111-9 on the preserved refreshed
top-20 panel. That is current-meta local evidence, not a new leaderboard-score
claim. The archive produced below is the competition submission artifact.


In [ ]:
from __future__ import annotations

import ast
import base64
import contextlib
import copy
import gzip
import hashlib
import importlib.util
import inspect
import io
import json
import os
from pathlib import Path
import py_compile
import sys
import tarfile
from typing import Any, Mapping
import zlib

try:
    import pandas as pd
except Exception:
    pd = None

WORK = Path("/kaggle/working")
if not WORK.exists():
    WORK = Path.cwd()
WORK.mkdir(parents=True, exist_ok=True)

MAIN_PATH = WORK / "main.py"
CONTROL_PATH = WORK / "control_c94.py"
ARCHIVE_PATH = WORK / "submission.tar.gz"
PYC_PATH = WORK / "main.validation.pyc"
TRANSITIONS_PATH = WORK / "adl_kaggriculture_transitions.jsonl"
MANIFEST_PATH = WORK / "c95_build_manifest.json"

COMPETITION_ID = 147734
EPISODE_STEPS = 720
TARGET_ENGINE = "1.32.6"
C95_LABEL = "c95_top20_wheat_counter"
C95_SHA256 = "489f5d197527f107027626cce79d850fd2ca90edd43d94384b849b6511e27bdb"
C95_BYTES = 75098
C94_LABEL = "c94_feed_first_fert_split"
C94_SHA256 = "7b0e5a7b9d18dc583f5789e50a54dca43561f6d08c1c616b4219bf50bcb8311f"
C94_BYTES = 75078

# Two paired seats over two seeds: a packaging/runtime veto, not a score claim.
RUN_PAIRED_GATE = True
PAIRED_GATE_SEEDS = (18590000, 18590001)

print({
    "work": str(WORK),
    "selected": C95_LABEL,
    "control": C94_LABEL,
    "episode_steps": EPISODE_STEPS,
    "internet_required": False,
})


## 1. Freeze and verify the exact C95 submission source

The payload is compressed only to keep the notebook below upload limits. The
decoded bytes, source length, and SHA-256 are fixed. Any policy edit causes this
cell to fail before an archive can be created.


In [ ]:
_C95_AGENT_B64_PARTS = [
    "..."
]

c95_raw = zlib.decompress(base64.b64decode("".join(_C95_AGENT_B64_PARTS)))
assert len(c95_raw) == C95_BYTES, (len(c95_raw), C95_BYTES)
c95_digest = hashlib.sha256(c95_raw).hexdigest()
assert c95_digest == C95_SHA256, (c95_digest, C95_SHA256)
compile(c95_raw, str(MAIN_PATH), "exec")
MAIN_PATH.write_bytes(c95_raw)

py_compile.compile(
    str(MAIN_PATH),
    cfile=str(PYC_PATH),
    doraise=True,
)
assert MAIN_PATH.read_bytes() == c95_raw
print({
    "main_py": str(MAIN_PATH),
    "bytes": MAIN_PATH.stat().st_size,
    "sha256": c95_digest,
    "compile": "passed",
})


## 2. Build the deterministic Kaggle submission archive

Only `main.py` is placed at the archive root. Gzip and tar timestamps are fixed
so repeated notebook runs produce the same tarball bytes.


In [ ]:
with ARCHIVE_PATH.open("wb") as raw_archive:
    with gzip.GzipFile(fileobj=raw_archive, mode="wb", mtime=0) as zipped:
        with tarfile.open(fileobj=zipped, mode="w") as archive:
            info = tarfile.TarInfo("main.py")
            info.size = len(c95_raw)
            info.mtime = 0
            info.mode = 0o644
            info.uid = 0
            info.gid = 0
            info.uname = ""
            info.gname = ""
            archive.addfile(info, io.BytesIO(c95_raw))

with tarfile.open(ARCHIVE_PATH, "r:gz") as archive:
    members = archive.getnames()
    archived_main = archive.extractfile("main.py").read()

assert members == ["main.py"], members
assert len(archived_main) == C95_BYTES
assert hashlib.sha256(archived_main).hexdigest() == C95_SHA256
archive_digest = hashlib.sha256(ARCHIVE_PATH.read_bytes()).hexdigest()
print({
    "submission": str(ARCHIVE_PATH),
    "archive_members": members,
    "archive_bytes": ARCHIVE_PATH.stat().st_size,
    "archive_sha256": archive_digest,
})


## 3. Embed the exact C94 veto control

C94 is not included in `submission.tar.gz`. It is preserved only as the parent
veto for paired evaluation and ADL transition capture.


In [ ]:
_C94_AGENT_B64_PARTS = [
    "..."
]

c94_raw = zlib.decompress(base64.b64decode("".join(_C94_AGENT_B64_PARTS)))
assert len(c94_raw) == C94_BYTES, (len(c94_raw), C94_BYTES)
c94_digest = hashlib.sha256(c94_raw).hexdigest()
assert c94_digest == C94_SHA256, (c94_digest, C94_SHA256)
compile(c94_raw, str(CONTROL_PATH), "exec")
CONTROL_PATH.write_bytes(c94_raw)
assert hashlib.sha256(CONTROL_PATH.read_bytes()).hexdigest() == C94_SHA256
print({"control": str(CONTROL_PATH), "bytes": len(c94_raw), "sha256": c94_digest})


## 4. File-runner and action-contract validation

Kaggle executes the last newly bound callable in a submitted Python file. This
cell executes the verified source in an isolated namespace, confirms that the
final callable delegates to the real C95 `agent`, and validates the function
signature without altering the source.


In [ ]:
def isolated_namespace(raw_source: bytes, tag: str) -> dict[str, Any]:
    namespace: dict[str, Any] = {
        "__name__": tag,
        "__file__": str(MAIN_PATH),
        "__builtins__": __builtins__,
    }
    exec(compile(raw_source, f"{tag}.py", "exec"), namespace, namespace)
    return namespace


c95_namespace = isolated_namespace(c95_raw, "c95_submission_validation")
new_callables = [
    (name, value)
    for name, value in c95_namespace.items()
    if callable(value) and not name.startswith("__")
]
assert new_callables, "no callable was defined by main.py"
selected_callable_name, selected_callable = new_callables[-1]
assert selected_callable_name == "c94_submission_agent", selected_callable_name
assert "agent" in c95_namespace and callable(c95_namespace["agent"])
signature = inspect.signature(selected_callable)
assert 1 <= len(signature.parameters) <= 2, signature

tree = ast.parse(c95_raw.decode("utf-8"), filename=str(MAIN_PATH))
top_level_functions = [node.name for node in tree.body if isinstance(node, ast.FunctionDef)]
assert top_level_functions[-1] == "c94_submission_agent"
print({
    "file_runner_callable": selected_callable_name,
    "signature": str(signature),
    "top_level_function_count": len(top_level_functions),
    "entrypoint_contract": "passed",
})


## 5. ADL transition recorder

The previous notebooks contained no machine-readable observed transitions.
This recorder captures compact, real `before_state → action → after_state`
records from evaluation games. It never wraps or modifies the submitted file.


In [ ]:
ADL_ITEMS = (
    "WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON",
    "EGG", "MILK", "WOOL", "FERTILIZER",
)
ADL_TILE_KEYS = (
    "COW", "SHEEP", "GOOSE", "WHEAT", "CARROT", "TOMATO",
    "STRAWBERRY", "MELON", "PASTURE", "COOP", "WEED",
)


def jsonable(value: Any) -> Any:
    if isinstance(value, Mapping):
        return {str(key): jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [jsonable(item) for item in value]
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    if hasattr(value, "item"):
        try:
            return jsonable(value.item())
        except Exception:
            pass
    if hasattr(value, "toJSON"):
        try:
            return jsonable(value.toJSON())
        except Exception:
            pass
    return str(value)


def stable_digest(value: Any) -> str:
    payload = json.dumps(jsonable(value), sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def get_value(value: Any, key: str, default: Any = None) -> Any:
    if isinstance(value, Mapping):
        return value.get(key, default)
    getter = getattr(value, "get", None)
    if callable(getter):
        try:
            return getter(key, default)
        except Exception:
            pass
    return getattr(value, key, default)


def tile_counts(farm: Mapping[str, Any]) -> dict[str, int]:
    counts = {key: 0 for key in ADL_TILE_KEYS}
    for row in get_value(farm, "tiles", []) or []:
        for tile in row or []:
            if not isinstance(tile, Mapping):
                continue
            for field in ("animal", "crop", "kind"):
                name = get_value(tile, field)
                if name in counts:
                    counts[name] += 1
                    break
    return counts


def score_proxy(obs: Any) -> float:
    player = int(get_value(obs, "player", 0) or 0)
    farms = list(get_value(obs, "farms", []) or [])
    farm = farms[player] if player < len(farms) else {}
    private = get_value(obs, "private", {}) or {}
    for container in (private, farm, obs):
        for key in ("money", "cash", "coins", "reward"):
            value = get_value(container, key)
            if isinstance(value, (int, float)):
                return float(value)
    return 0.0


def adl_state(obs: Any) -> dict[str, Any]:
    player = 1 if int(get_value(obs, "player", 0) or 0) == 1 else 0
    farms = list(get_value(obs, "farms", []) or [])
    farm = farms[player] if player < len(farms) else {}
    private = get_value(obs, "private", {}) or {}
    market = get_value(obs, "market", {}) or {}
    town = get_value(obs, "town", {}) or {}
    counts = tile_counts(farm)
    shed = {item: float(get_value(get_value(private, "shed", {}) or {}, item, 0) or 0) for item in ADL_ITEMS}
    inventory = {item: float(get_value(get_value(market, "inventory", {}) or {}, item, 0) or 0) for item in ADL_ITEMS}
    prices = {item: float(get_value(get_value(market, "prices", {}) or {}, item, 0) or 0) for item in ADL_ITEMS}
    step = int(get_value(obs, "step", 0) or 0)
    hands = len(get_value(farm, "hands", []) or [])
    money = score_proxy(obs)
    vector = [
        step / float(EPISODE_STEPS),
        int(get_value(obs, "day", step // 24) or 0) / 30.0,
        int(get_value(obs, "hour", step % 24) or 0) / 24.0,
        money / 200000.0,
        hands / 20.0,
        *[counts[key] / 100.0 for key in ADL_TILE_KEYS],
        *[shed[item] / 1000.0 for item in ADL_ITEMS],
        *[inventory[item] / 20000.0 for item in ADL_ITEMS],
        *[prices[item] / 2000.0 for item in ADL_ITEMS],
    ]
    return {
        "state": vector,
        "step": step,
        "day": int(get_value(obs, "day", step // 24) or 0),
        "hour": int(get_value(obs, "hour", step % 24) or 0),
        "player": player,
        "money": money,
        "hands": hands,
        "unlocked_quadrants": sorted(str(value) for value in (get_value(farm, "unlocked_quadrants", []) or [])),
        "tiles": counts,
        "shed": shed,
        "market_inventory": inventory,
        "market_prices": prices,
        "unlocked_shops": sorted(str(value) for value in (get_value(town, "unlocked_shops", []) or [])),
    }


def causal_state(state: Mapping[str, Any]) -> dict[str, Any]:
    return {
        key: value
        for key, value in state.items()
        if key not in {"state", "step", "day", "hour"}
    }


def call_agent(agent_callable: Any, obs: Any, config: Any = None) -> Any:
    try:
        return agent_callable(obs, config)
    except TypeError:
        return agent_callable(obs)


class ADLTransitionRecorder:
    def __init__(
        self,
        agent_callable: Any,
        *,
        game_id: str,
        episode_id: str,
        strategy: str,
        seat: int,
        opponent: str,
        environment_files: list[dict[str, Any]],
    ) -> None:
        self.agent_callable = agent_callable
        self.game_id = game_id
        self.episode_id = episode_id
        self.strategy = strategy
        self.seat = int(seat)
        self.opponent = opponent
        self.environment_files = copy.deepcopy(environment_files)
        self.records: list[dict[str, Any]] = []
        self.pending: dict[str, Any] | None = None
        self.decision_index = 0

    def _emit(self, after_observation: Any, *, done: bool) -> None:
        if self.pending is None:
            return
        after_state = adl_state(after_observation)
        before_state = self.pending["before_state"]
        score_before = float(self.pending["score_before"])
        score_after = float(after_state["money"])
        record = {
            "game_id": self.game_id,
            "episode_id": self.episode_id,
            "step_index": int(self.pending["step_index"]),
            "decision_id": self.pending["decision_id"],
            "before_state": before_state,
            "after_state": after_state,
            "target_state": {
                "objective": "maximize head-to-head win probability after 720 turns",
                "episode_steps": EPISODE_STEPS,
            },
            "action": self.pending["action"],
            "strategy": self.strategy,
            "score_before": score_before,
            "score_after": score_after,
            "reward": score_after - score_before,
            "level_complete": False,
            "done": bool(done),
            "success": False,
            "invalid": False,
            "no_impact": stable_digest(causal_state(before_state)) == stable_digest(causal_state(after_state)),
            "environment_files": self.environment_files,
            "context": {
                "seat": self.seat,
                "opponent": self.opponent,
                "episode_step": int(self.pending["step_index"]),
                "remaining_actions": max(0, EPISODE_STEPS - int(self.pending["step_index"]) - 1),
            },
        }
        self.records.append(record)
        self.pending = None

    def __call__(self, obs: Any, config: Any = None) -> Any:
        self._emit(obs, done=False)
        state = adl_state(obs)
        action = jsonable(call_agent(self.agent_callable, obs, config))
        step = int(state["step"])
        self.pending = {
            "decision_id": f"{self.game_id}:{self.seat}:{step}:{self.decision_index}",
            "step_index": step,
            "before_state": state,
            "score_before": float(state["money"]),
            "action": action,
        }
        self.decision_index += 1
        return action

    def close(self, final_observation: Any, *, done: bool = True) -> list[dict[str, Any]]:
        self._emit(final_observation, done=done)
        return list(self.records)


# Deterministic unit probe for the transition contract.
def _probe_agent(obs: Any, config: Any = None) -> dict[str, Any]:
    return {"farmer": ["PASS"], "hands": [], "market": []}


probe_files = [{"name": "main.py", "sha256": C95_SHA256, "bytes": C95_BYTES}]
probe = ADLTransitionRecorder(
    _probe_agent,
    game_id="contract-probe",
    episode_id="contract-probe",
    strategy="contract_probe",
    seat=0,
    opponent="none",
    environment_files=probe_files,
)
probe({"step": 0, "player": 0, "farms": [{"tiles": [], "hands": []}], "private": {"money": 3000}})
probe({"step": 1, "player": 0, "farms": [{"tiles": [], "hands": []}], "private": {"money": 3001}})
assert len(probe.records) == 1
assert probe.records[0]["reward"] == 1.0
assert {"before_state", "after_state", "action", "environment_files"} <= set(probe.records[0])
print({"adl_transition_contract": "passed", "probe_records": len(probe.records)})


## 6. Competition-engine smoke and paired C95/C94 veto

Artifact generation never depends on `kaggle_environments`. When the competition
engine is installed, this cell runs the exact C95 file path against `starter`,
then runs C95 and C94 in both seats over the fixed gate seeds while recording
real transitions. If the engine is absent, the validated submission archive is
still produced and the engine-dependent gate is reported as skipped.


In [ ]:
ENGINE_AVAILABLE = False
ENGINE_VERSION = None
ENGINE_ERROR = None
SMOKE_RESULT = None
PAIRED_RESULTS: list[dict[str, Any]] = []
ADL_RECORDS: list[dict[str, Any]] = []

try:
    import kaggle_environments
    from kaggle_environments import make

    ENGINE_AVAILABLE = True
    ENGINE_VERSION = getattr(kaggle_environments, "__version__", "unknown")
except Exception as exc:
    ENGINE_ERROR = f"{type(exc).__name__}: {exc}"


def load_agent(path: Path, tag: str) -> Any:
    spec = importlib.util.spec_from_file_location(tag, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"could not load {path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    if not callable(getattr(module, "agent", None)):
        raise RuntimeError(f"{path} has no callable agent")
    return module.agent


def valid_action(action: Any) -> bool:
    return (
        isinstance(action, Mapping)
        and isinstance(action.get("farmer"), list)
        and bool(action.get("farmer"))
        and isinstance(action.get("hands"), list)
        and all(isinstance(order, list) and bool(order) for order in action.get("hands"))
        and isinstance(action.get("market"), list)
        and len(action.get("market")) <= 10
        and all(isinstance(order, list) and bool(order) for order in action.get("market"))
    )


def state_observation(state: Any) -> Any:
    value = getattr(state, "observation", None)
    return value if value is not None else get_value(state, "observation", {})


if ENGINE_AVAILABLE:
    smoke_env = make(
        "kaggriculture",
        configuration={"episodeSteps": EPISODE_STEPS, "seed": 18590000},
        debug=False,
    )
    smoke_env.run([str(MAIN_PATH), "starter"])
    smoke_final = smoke_env.steps[-1]
    smoke_status = [str(state.status) for state in smoke_final]
    smoke_rewards = [float(state.reward) for state in smoke_final]
    smoke_actions = [
        state.action
        for frame in smoke_env.steps[1:]
        for state in frame
        if state.action is not None
    ]
    assert len(smoke_env.steps) == EPISODE_STEPS
    assert smoke_status == ["DONE", "DONE"], smoke_status
    assert smoke_actions and all(valid_action(action) for action in smoke_actions)
    SMOKE_RESULT = {
        "status": smoke_status,
        "rewards": smoke_rewards,
        "validated_actions": len(smoke_actions),
    }

    if RUN_PAIRED_GATE:
        environment_files = [
            {"name": "main.py", "sha256": C95_SHA256, "bytes": C95_BYTES},
            {"name": "control_c94.py", "sha256": C94_SHA256, "bytes": C94_BYTES},
            {"name": "kaggriculture", "competition_id": COMPETITION_ID, "engine": ENGINE_VERSION},
        ]
        for seed in PAIRED_GATE_SEEDS:
            for c95_seat in (0, 1):
                game_id = f"c95-c94-seed-{seed}-c95seat-{c95_seat}"
                c95_agent = load_agent(MAIN_PATH, f"c95_{seed}_{c95_seat}")
                c94_agent = load_agent(CONTROL_PATH, f"c94_{seed}_{c95_seat}")
                c95_recorder = ADLTransitionRecorder(
                    c95_agent,
                    game_id=game_id,
                    episode_id=game_id,
                    strategy=C95_LABEL,
                    seat=c95_seat,
                    opponent=C94_LABEL,
                    environment_files=environment_files,
                )
                c94_recorder = ADLTransitionRecorder(
                    c94_agent,
                    game_id=game_id,
                    episode_id=game_id,
                    strategy=C94_LABEL,
                    seat=1 - c95_seat,
                    opponent=C95_LABEL,
                    environment_files=environment_files,
                )
                agents = [None, None]
                agents[c95_seat] = c95_recorder
                agents[1 - c95_seat] = c94_recorder
                env = make(
                    "kaggriculture",
                    configuration={"episodeSteps": EPISODE_STEPS, "seed": int(seed)},
                    debug=False,
                )
                env.run(agents)
                final = env.steps[-1]
                statuses = [str(state.status) for state in final]
                rewards = [float(state.reward) for state in final]
                assert len(env.steps) == EPISODE_STEPS
                assert statuses == ["DONE", "DONE"], statuses
                c95_reward = rewards[c95_seat]
                c94_reward = rewards[1 - c95_seat]
                PAIRED_RESULTS.append({
                    "seed": int(seed),
                    "c95_seat": c95_seat,
                    "c95_reward": c95_reward,
                    "c94_reward": c94_reward,
                    "margin": c95_reward - c94_reward,
                    "c95_win": int(c95_reward > c94_reward),
                    "tie": int(c95_reward == c94_reward),
                    "statuses": statuses,
                })
                c95_final_obs = state_observation(final[c95_seat])
                c94_final_obs = state_observation(final[1 - c95_seat])
                ADL_RECORDS.extend(c95_recorder.close(c95_final_obs, done=True))
                ADL_RECORDS.extend(c94_recorder.close(c94_final_obs, done=True))

        with TRANSITIONS_PATH.open("w", encoding="utf-8", newline="\n") as handle:
            for record in ADL_RECORDS:
                handle.write(json.dumps(record, sort_keys=True, separators=(",", ":"), ensure_ascii=False))
                handle.write("\n")

    print({
        "engine_available": True,
        "engine_version": ENGINE_VERSION,
        "target_engine": TARGET_ENGINE,
        "smoke": SMOKE_RESULT,
        "paired_games": len(PAIRED_RESULTS),
        "adl_transitions": len(ADL_RECORDS),
    })
    if PAIRED_RESULTS:
        if pd is not None:
            print(pd.DataFrame(PAIRED_RESULTS).to_string(index=False))
        else:
            print(json.dumps(PAIRED_RESULTS, indent=2))
else:
    print({
        "engine_available": False,
        "engine_gate": "skipped",
        "reason": ENGINE_ERROR,
        "artifact_build": "passed independently",
    })


## 7. Final manifest and promotion guard

C95 is the only policy placed in the archive. C94 remains the veto control.
The notebook records the exact identity, validation boundary, paired result,
and ADL corpus path so later scores cannot be attributed to a filename-only
change.


In [ ]:
paired_summary = None
if PAIRED_RESULTS:
    wins = sum(row["c95_win"] for row in PAIRED_RESULTS)
    ties = sum(row["tie"] for row in PAIRED_RESULTS)
    losses = len(PAIRED_RESULTS) - wins - ties
    margins = [float(row["margin"]) for row in PAIRED_RESULTS]
    paired_summary = {
        "games": len(PAIRED_RESULTS),
        "c95_wins": wins,
        "ties": ties,
        "c95_losses": losses,
        "mean_margin": sum(margins) / len(margins),
        "seeds": list(PAIRED_GATE_SEEDS),
        "both_seats": True,
        "claim": "runtime veto only; not a leaderboard-score claim",
    }

manifest = {
    "artifact": C95_LABEL,
    "competition_id": COMPETITION_ID,
    "submission_file": str(ARCHIVE_PATH),
    "archive_sha256": hashlib.sha256(ARCHIVE_PATH.read_bytes()).hexdigest(),
    "archive_members": ["main.py"],
    "main_py": {
        "path": str(MAIN_PATH),
        "bytes": C95_BYTES,
        "sha256": C95_SHA256,
        "entrypoint_selected_by_file_runner": selected_callable_name,
    },
    "offline_veto": {
        "label": C94_LABEL,
        "bytes": C94_BYTES,
        "sha256": C94_SHA256,
    },
    "policy_change_from_frozen_c95": None,
    "instrumentation_inside_submission": False,
    "engine": {
        "available": ENGINE_AVAILABLE,
        "version": ENGINE_VERSION,
        "target": TARGET_ENGINE,
        "smoke": SMOKE_RESULT,
    },
    "paired_gate": paired_summary,
    "adl": {
        "transition_count": len(ADL_RECORDS),
        "transition_file": str(TRANSITIONS_PATH) if ADL_RECORDS else None,
        "environment_files_included": bool(ADL_RECORDS),
        "synthetic_transitions": 0,
    },
    "score_claim": None,
    "selection_evidence": {
        "c95_refreshed_top20": "112-8 over 120 preserved local games",
        "c94_refreshed_top20": "111-9 over the same preserved local games",
        "c94_broad_baseline": "174-6 held-out; 88-14, BT 1837 in preserved final",
    },
}
MANIFEST_PATH.write_text(
    json.dumps(manifest, sort_keys=True, indent=2) + "\n",
    encoding="utf-8",
)
assert ARCHIVE_PATH.is_file() and ARCHIVE_PATH.stat().st_size > 0
assert MAIN_PATH.is_file() and hashlib.sha256(MAIN_PATH.read_bytes()).hexdigest() == C95_SHA256
print(json.dumps(manifest, sort_keys=True, indent=2))
print(f"READY: submit {ARCHIVE_PATH}")


## Submission

Run all cells. Download `/kaggle/working/submission.tar.gz` and submit that
tarball to Kaggriculture. Its sole archive member is the exact validated C95
`main.py`. The ADL JSONL and manifest are evaluation evidence, not submission
members.
